# Biohub: thử tạo submission từ checkpoint Linajea

Notebook đọc test .zarr, không cần test .geff. Nó tạo submission.csv với node và temporal edge. Chạy thử một sample trước rồi chạy toàn bộ test. Đây là bản kiểm tra kỹ thuật từ checkpoint seed 42 đã train trên một sample, chưa chứng minh khả năng tổng quát hóa sang embryo test.

Cuộc thi ghi hạn cuối 29/09/2026. Sau hạn này, Kaggle chỉ có thể chấp nhận nếu giao diện tài khoản bạn còn cho phép Late Submission. Notebook submission phải chạy khi Internet tắt; hãy gắn file linajea_submit_source.zip, checkpoint + config, và các dependency còn thiếu như Kaggle Inputs. Notebook này không clone GitHub lúc chạy.


In [ ]:
from pathlib import Path
import sys, subprocess
from zipfile import ZipFile

TEST_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/test')
SOURCE_ZIP = Path('/kaggle/input/REPLACE-SOURCE-DATASET/linajea_submit_source.zip')
SOURCE_REPO = Path('/kaggle/working/linajea_source/linajea')
RUN_DIR = Path('/kaggle/input/REPLACE-CHECKPOINT-DATASET/seed_42')
CHECKPOINT = RUN_DIR / 'train' / 'train_net_checkpoint_1500'
CONFIG = RUN_DIR / 'train' / 'config.toml'
OUT = Path('/kaggle/working/biohub_submission')

# Sửa SOURCE_ZIP và RUN_DIR theo tên Input thực tế.
for path in (TEST_DIR, SOURCE_ZIP, CHECKPOINT, CONFIG):
    if not path.exists():
        raise FileNotFoundError(f'Missing Input: {path}')

SOURCE_REPO.parent.mkdir(parents=True, exist_ok=True)
with ZipFile(SOURCE_ZIP) as archive:
    archive.extractall(SOURCE_REPO.parent)
assert (SOURCE_REPO / 'linajea' / 'biohub_submission.py').is_file()
sys.path.insert(0, str(SOURCE_REPO))
try:
    import torch, zarr, scipy
    from linajea.biohub_submission import COLUMNS
    from linajea.training.torch_model import UnetModelWrapper
except ImportError as error:
    raise RuntimeError(
        'Kaggle Internet-off run lacks a dependency. Attach an offline '
        'wheelhouse as Input and install it locally before this cell.'
    ) from error
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU')
samples = sorted(TEST_DIR.glob('*.zarr'))
if not samples:
    raise FileNotFoundError(f'No test Zarr in {TEST_DIR}')
print('Test datasets:', len(samples), 'first:', samples[0].stem)
print('Checkpoint:', CHECKPOINT)
print('GPU:', torch.cuda.get_device_name(0))


## 1. Kiểm tra một sample

Cell này tạo smoke_NAME.csv, không phải file nộp. Nó có thể chạy lâu vì suy luận toàn bộ 100 frame của sample.


In [ ]:
sample = samples[0].stem
smoke = [
    sys.executable, '-m', 'run_scripts.submit_biohub',
    '--data-dir', str(TEST_DIR),
    '--config', str(CONFIG),
    '--checkpoint', str(CHECKPOINT),
    '--output-dir', str(OUT),
    '--device', 'cuda',
    '--selection-constant', '7',
    '--track-cost', '7',
    '--sample', sample,
]
subprocess.run(smoke, cwd=SOURCE_REPO, check=True)
smoke_csv = OUT / f'smoke_{sample}.csv'
print('Smoke CSV:', smoke_csv, 'bytes:', smoke_csv.stat().st_size)


## 2. Tạo file cho toàn bộ test

Chạy cell này chỉ sau khi smoke test đạt. Nó có thể mất nhiều giờ. Kaggle Code Competition giới hạn 12 giờ; nếu vượt thời gian, notebook không thể submit bằng cách này. Cache graph đã hoàn thành được dùng lại khi chạy tiếp cùng settings.


In [ ]:
full = [
    sys.executable, '-m', 'run_scripts.submit_biohub',
    '--data-dir', str(TEST_DIR),
    '--config', str(CONFIG),
    '--checkpoint', str(CHECKPOINT),
    '--output-dir', str(OUT),
    '--device', 'cuda',
    '--selection-constant', '7',
    '--track-cost', '7',
]
subprocess.run(full, cwd=SOURCE_REPO, check=True)
submission = OUT / 'submission.csv'
import shutil
final_path = Path('/kaggle/working/submission.csv')
shutil.copy2(submission, final_path)
print('Submission:', final_path, 'bytes:', final_path.stat().st_size)


## 3. Kiểm tra CSV cuối

Mỗi test dataset phải xuất hiện. ID node chỉ cần duy nhất trong dataset; cột id phải liên tiếp trên toàn CSV.


In [ ]:
import csv
from collections import defaultdict

seen = defaultdict(set)
nodes = defaultdict(set)
edges = defaultdict(list)
with final_path.open(newline='', encoding='utf-8') as stream:
    reader = csv.DictReader(stream)
    assert tuple(reader.fieldnames) == COLUMNS
    for expected_id, row in enumerate(reader):
        assert int(row['id']) == expected_id
        dataset = row['dataset']
        seen[dataset].add(row['row_type'])
        if row['row_type'] == 'node':
            node_id = int(row['node_id'])
            assert node_id not in nodes[dataset]
            nodes[dataset].add(node_id)
            assert int(row['source_id']) == int(row['target_id']) == -1
        elif row['row_type'] == 'edge':
            edges[dataset].append((int(row['source_id']),
                                   int(row['target_id'])))
            assert all(int(row[c]) == -1 for c in ('node_id','t','z','y','x'))
        else:
            raise AssertionError(f'Bad row_type: {row["row_type"]}')
expected = {path.stem for path in samples}
assert set(seen) == expected, (expected - set(seen), set(seen) - expected)
for dataset, links in edges.items():
    assert all(a in nodes[dataset] and b in nodes[dataset] for a, b in links)
print('CSV validation passed:', len(expected), 'datasets,',
      sum(map(len, nodes.values())), 'nodes,',
      sum(map(len, edges.values())), 'edges')
print('File for Kaggle:', final_path)
